In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import json
from time import strftime, sleep
from xmltodict import parse
from datetime import datetime, date
import re

In [2]:
def get_vacancies(query, area=40):
    url = "https://api.hh.kz/vacancies"
    all_items = []
    page = 0
    while True:
        params = {"per_page": 100, "text": query, "area": area, "page": page}
        data = requests.get(url, params=params).json()
        all_items.extend(data['items'])
        if page >= data['pages']-1:
            break
        page += 1
    return {'items': all_items}

In [3]:
queries = [
    # ML/DS
    "ML engineer", "Data analyst", "Data scientist", "Data science",
    "Аналитик данных", "мл инженер", "data engineer", "дата инженер",
    "NLP engineer", "AI engineer", "machine learning engineer", "machine learning", "computer vision engineer",
    'MLOps',

    # Backend
    "Python developer", "python разработчик", "Backend Python",
    "Django developer", "FastAPI developer", "Flask developer",
    "Software engineer Python", 'java developer', '.NET developer',
    'Node.js developer', 'Backend разработчик', 'Software engineer',
    'php developer', 'Go developer', 'C# developer',

    # Frontend/mobile
    "Frontend developer", "React developer", "Vue developer",
    "iOS developer", "Android developer", "Flutter developer", "Full stack developer",
    'Разработчик мобильных игр',

    # DevOps
    "DevOps engineer", "Cloud engineer", "Kubernetes", "SRE engineer",
    "Системный администратор", "Network engineer",

    # QA
    "QA engineer", "Тестировщик", "Automation QA", "SDET",

    # Analytics/management
    "Системный аналитик", "Бизнес-аналитик IT", "Product manager IT",
    "Scrum master", "IT project manager",

    # Security / DB
    "Cybersecurity engineer", "Information security", "DBA", "Database administrator",
]

IT_ROLE_KEYWORDS = {
    "разработчик", "developer", "engineer", "аналитик", "analyst",
    "devops", "mlops", "qa", "тестировщик", "администратор", "data",
    "architect", "архитектор", "ios", "android", "frontend", "backend",
    "fullstack", "security", "scrum", "product manager", "pm", "dba",
    "программист", "scientist", "ml", "ai",
}

NON_IT_BLOCKLIST = {
    "бухгалтер", "accountant", "учитель", "teacher", "воспитатель",
    "юрист", "lawyer", "врач", "doctor", "медсестра", "менеджер по продажам",
    "sales manager", "кассир", "повар", "cook", "водитель", "driver",
    "логист", "economist", "экономист",
}

In [4]:
all_vacancies = []
seen_ids = set()

for query in queries:
    data = get_vacancies(query)
    for item in data['items']:
        if item['id'] in seen_ids:
            continue

        # First filter to check names of vacancies
        name = item.get('name', '').lower()
        if any(bad in name for bad in NON_IT_BLOCKLIST):
            continue

        # filter 2 - at least 1 it word should be in professional role
        roles_text = " ".join(
            r.get('name', '').lower()
            for r in item.get('professional_roles', [])
        )
        combined = name + " " + roles_text
        if not any(keyword in combined for keyword in IT_ROLE_KEYWORDS):
            continue

        all_vacancies.append(item)
        seen_ids.add(item['id'])

In [5]:
def get_details(vacancy):
    vacancy_id = vacancy['id']
    detail = requests.get(f"https://api.hh.kz/vacancies/{vacancy_id}").json()
    sleep(0.5)
    description_html = detail['description']
    clean_text = BeautifulSoup(description_html, "html.parser").get_text()
    return clean_text

def get_currency(title):
    today_str = strftime("%d.%m.%Y")
    url = f"https://nationalbank.kz/rss/get_rates.cfm?fdate={today_str}"
    content = requests.get(url).content
    parsed = parse(content)
    items = parsed['rates']['item']
    currency = next(item for item in items if item['title']==title)
    currency_rate = currency['description']
    return float(currency_rate)

def flatten_vacancy(v):
    sal_range = v.get('salary_range') or {}
    sal_old   = v.get('salary') or {}

    return {
        'id': v.get('id'),
        'salary_from': sal_range.get('from') or sal_old.get('from'),
        'salary_to':   sal_range.get('to')   or sal_old.get('to'),

        'area':         v.get('area', {}).get('name'),
        'experience':   v.get('experience', {}).get('id'),
        'employment':   v.get('employment', {}).get('id'),
        'schedule':     v.get('schedule', {}).get('id'),
        'work_format':  [f['id'] for f in v.get('work_format', [])],
        'professional_role': [r['name'] for r in v.get('professional_roles', [])],

        # Boolean
        'is_premium':       v.get('premium', False),
        'has_test':         v.get('has_test', False),
        'accept_temporary': v.get('accept_temporary', False),
        'is_gross':         sal_range.get('gross', False),
        'has_usd_in_description': False,

        'published_at': v.get('published_at'),

        'description': None,
        'name': v.get('name'),
    }

In [6]:
data = []
rates_cache = {'KZT': 1.0}
currency_aliases = {'RUR': 'RUB'}
USD_PAT = re.compile(r'usd|\$|доллар|dollar', re.IGNORECASE)

for i in range(len(all_vacancies)):
    data.append(flatten_vacancy(all_vacancies[i]))
    description = get_details(all_vacancies[i])

    data[i]['description'] = description
    data[i]['has_usd_in_description'] = bool(USD_PAT.search(description or ''))

    sal_obj = all_vacancies[i].get('salary_range') or all_vacancies[i].get('salary') or {}
    currency = sal_obj.get('currency')
    bank_currency = currency_aliases.get(currency, currency)
    if bank_currency and bank_currency != 'KZT':
        if bank_currency not in rates_cache:
            rates_cache[bank_currency] = get_currency(bank_currency)
        rate = rates_cache[bank_currency]

        if data[i]['salary_from'] is not None:
            data[i]['salary_from'] *= rate
        if data[i]['salary_to'] is not None:
            data[i]['salary_to'] *= rate

    published_raw = data[i]['published_at']
    if published_raw:
        data[i]['published_at'] = datetime.fromisoformat(published_raw)

In [8]:
print(data[0].keys())
data[0]

dict_keys(['id', 'salary_from', 'salary_to', 'area', 'experience', 'employment', 'schedule', 'work_format', 'professional_role', 'is_premium', 'has_test', 'accept_temporary', 'is_gross', 'has_usd_in_description', 'published_at', 'description', 'name'])


{'id': '132031395',
 'salary_from': 700000,
 'salary_to': 1200000,
 'area': 'Алматы',
 'experience': 'between1And3',
 'employment': 'full',
 'schedule': 'fullDay',
 'work_format': ['ON_SITE', 'HYBRID'],
 'professional_role': ['Дата-сайентист'],
 'is_premium': False,
 'has_test': False,
 'accept_temporary': False,
 'is_gross': False,
 'has_usd_in_description': False,
 'published_at': datetime.datetime(2026, 4, 10, 16, 2, 39, tzinfo=datetime.timezone(datetime.timedelta(seconds=10800))),
 'description': 'Мы разрабатываем и развиваем продукт в сфере умного дома и безопасности. Проходим путь от разработки hardware решений до написание своих software решений. На данный момент усиляем команду в направлении сomputer vision: определение событий и контекста в видео потоке и отправка триггера на интерфейс. Обязанности:  Разработка и внедрение моделей компьютерного зрения для анализа видеопотока Распознавание действий (action recognition) и сценариев поведения Определение контекста происходящего (

In [9]:
df = pd.DataFrame(data)
df.head(2)

,id,salary_from,salary_to,area,experience,employment,schedule,work_format,professional_role,is_premium,has_test,accept_temporary,is_gross,has_usd_in_description,published_at,description,name
0,132031395,700000.0,1200000.0,Алматы,between1And3,full,fullDay,"[ON_SITE, HYBRID]",[Дата-сайентист],False,False,False,False,False,2026-04-10 16:02:39+03:00,Мы разрабатываем и развиваем продукт в сфере у...,ML Engineer / CV Engineer
1,131823118,9462.6,NaN,Астана,between1And3,full,remote,[REMOTE],[Специалист технической поддержки],False,False,True,False,True,2026-04-03 15:03:43+03:00,"Привет! Мы, ""Aspirity Solution"", студия разраб...",Customer Support Engineer


In [10]:
df.to_csv(f"../data/raw/vacancies_{date.today()}.csv", index=False)